# 因果連鎖分析: 気候 → 農業 → 人口 → 財政 → 社会不安

## 概要
本ノートブックでは、3世紀の危機（AD235-284年）における以下の因果連鎖を検証します。

```
気候変動 → 農業生産力低下 → 人口減少 → 財政悪化 → 社会不安
```

## 分析手法
1. 探索的データ分析（EDA）
2. 相関分析
3. クロス相関分析（タイムラグ検出）
4. グレンジャー因果性検定
5. VARモデル分析

## 1. 環境設定

In [ ]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定
plt.rcParams['font.family'] = 'MS Gothic'  # Windows
# plt.rcParams['font.family'] = 'Hiragino Sans'  # macOS

# プロジェクトパスの設定
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# カスタムモジュールのインポート
from src.data_loader import CausalChainDataLoader
from src.causal_analysis import CausalChainAnalyzer
from src.config import ANALYSIS_PERIOD, FIGURES_DIR

print(f"プロジェクトルート: {project_root}")
print(f"分析期間: AD {ANALYSIS_PERIOD['start']} - {ANALYSIS_PERIOD['end']}")

## 2. データの読み込み

In [ ]:
# データローダーの初期化
loader = CausalChainDataLoader()

# 各データソースの読み込み状況を確認
print("=== データ読み込み ===")

# 気候データ
print("\n1. 気候データ:")
climate = loader.load_climate_data()
print(f"   期間: {climate['year'].min()} - {climate['year'].max()}")
print(f"   件数: {len(climate)}")

# 難破船データ
print("\n2. 難破船データ:")
shipwrecks = loader.load_shipwreck_data()
print(f"   件数: {len(shipwrecks)}")

# 碑文データ
print("\n3. 碑文データ:")
inscriptions = loader.load_inscription_data()
print(f"   件数: {len(inscriptions)}")

# 銀含有率データ
print("\n4. 銀含有率データ:")
silver = loader.load_silver_data()
print(f"   期間: {silver['year'].min()} - {silver['year'].max()}")
print(f"   件数: {len(silver)}")

# Seshatデータ
print("\n5. Seshat不安定データ:")
seshat = loader.load_seshat_data()
print(f"   件数: {len(seshat)}")

In [ ]:
# 統合データセットの作成
print("=== 統合データセット作成 ===")
unified = loader.create_unified_dataset()

print(f"\n統合データ:")
print(f"  期間: AD {unified['year'].min()} - {unified['year'].max()}")
print(f"  件数: {len(unified)} (10年単位)")
print(f"  カラム: {list(unified.columns)}")

# データプレビュー
print("\nデータプレビュー:")
unified

In [ ]:
# 記述統計
print("=== 記述統計 ===")
unified.describe().round(2)

## 3. 探索的データ分析（EDA）

In [ ]:
# 因果連鎖分析器の初期化
analyzer = CausalChainAnalyzer(unified)

# 時系列プロット
fig = analyzer.plot_time_series(figsize=(14, 12))
plt.show()

In [ ]:
# 個別変数の時系列プロット（詳細版）
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

variables = ['climate', 'trade', 'population', 'fiscal', 'instability']
colors = ['steelblue', 'green', 'orange', 'red', 'purple']
titles = ['気候（気温偏差）', '交易（難破船数）', '人口（碑文数）', '財政（銀含有率%）', '不安定指標']

crisis_start = ANALYSIS_PERIOD['crisis_start']
crisis_end = ANALYSIS_PERIOD['crisis_end']

for i, (var, color, title) in enumerate(zip(variables, colors, titles)):
    ax = axes[i]
    ax.plot(unified['year'], unified[var], color=color, linewidth=2, marker='o', markersize=4)
    ax.axvspan(crisis_start, crisis_end, alpha=0.2, color='red')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('年 (CE)')
    ax.grid(True, alpha=0.3)
    
    # 3世紀の危機をラベル
    if i == 0:
        ax.annotate('3世紀の危機', xy=(260, ax.get_ylim()[1]*0.9), fontsize=10, color='red')

# 最後のサブプロットを非表示
axes[-1].axis('off')

plt.suptitle('因果連鎖変数の時系列（AD 1-300年）', fontsize=14)
plt.tight_layout()
plt.show()

## 4. 相関分析

In [ ]:
# 相関行列の計算
corr_matrix = analyzer.compute_correlation_matrix()

print("=== 相関行列 ===")
print(corr_matrix.round(3))

In [ ]:
# 相関行列のヒートマップ
fig = analyzer.plot_correlation_heatmap(figsize=(10, 8))
plt.show()

In [ ]:
# ペアプロット（散布図行列）
var_labels = {
    'climate': '気候',
    'trade': '交易',
    'population': '人口',
    'fiscal': '財政',
    'instability': '不安定'
}

plot_data = unified[['climate', 'trade', 'population', 'fiscal', 'instability']].copy()
plot_data.columns = [var_labels[c] for c in plot_data.columns]

g = sns.pairplot(plot_data, diag_kind='kde', plot_kws={'alpha': 0.6})
g.fig.suptitle('変数間の散布図行列', y=1.02)
plt.show()

## 5. クロス相関分析（タイムラグ検出）

In [ ]:
# 因果連鎖の各ステップのクロス相関
fig = analyzer.plot_cross_correlation(figsize=(14, 10))
plt.show()

In [ ]:
# 最適ラグの特定
chain = [
    ('climate', 'trade'),
    ('trade', 'population'),
    ('population', 'fiscal'),
    ('fiscal', 'instability')
]

print("=== 最適ラグ分析 ===")
print(f"{'原因':<15} {'結果':<15} {'最適ラグ':<10} {'相関係数':<10} {'解釈'}")
print("-" * 70)

for cause, effect in chain:
    result = analyzer.find_optimal_lag(cause, effect)
    print(f"{analyzer.var_labels[cause]:<15} {analyzer.var_labels[effect]:<15} "
          f"{result['optimal_lag']:<10} {result['max_correlation']:>8.3f}   {result['direction']}")

## 6. 定常性検定（ADF検定）

In [ ]:
# 全変数の定常性検定
print("=== 定常性検定（ADF検定）===")
print("H0: 単位根あり（非定常）")
print("H1: 単位根なし（定常）")
print("p < 0.05 で定常と判定\n")

stationarity_results = analyzer.test_all_stationarity()
print(stationarity_results.to_string(index=False))

## 7. グレンジャー因果性検定

In [ ]:
# 因果連鎖の各ステップを検定
print("=== グレンジャー因果性検定 ===")
print("H0: XはYをグレンジャー原因しない")
print("H1: XはYをグレンジャー原因する")
print("p < 0.05 で因果関係ありと判定\n")

causal_results = analyzer.test_causal_chain()
print(causal_results.to_string(index=False))

In [ ]:
# 各ステップの詳細なp値（ラグごと）
print("=== ラグごとのp値（詳細）===")

for cause, effect in chain:
    try:
        result = analyzer.test_granger_causality(cause, effect)
        print(f"\n{analyzer.var_labels[cause]} → {analyzer.var_labels[effect]}:")
        for lag, p in result['p_values'].items():
            sig = '*' if p < 0.05 else ''
            print(f"  ラグ {lag}: p = {p:.4f} {sig}")
    except Exception as e:
        print(f"\n{cause} → {effect}: エラー - {e}")

## 8. VARモデル分析

In [ ]:
# VARモデルの推定
print("=== VARモデル推定 ===")
try:
    var_model = analyzer.fit_var_model(max_lag=5)
    if var_model is not None:
        print(f"最適ラグ（AIC）: {analyzer.results['var_optimal_lag']}")
        print("\nモデル要約:")
        print(var_model.summary())
except Exception as e:
    print(f"VARモデル推定エラー: {e}")

## 9. 結果のまとめ

In [ ]:
# 分析レポートの生成
report = analyzer.generate_report()
print(report)

In [ ]:
# 因果連鎖の可視化（最終結果）
fig, ax = plt.subplots(figsize=(14, 4))

# ノードの位置
nodes = {
    'climate': (0.1, 0.5),
    'trade': (0.3, 0.5),
    'population': (0.5, 0.5),
    'fiscal': (0.7, 0.5),
    'instability': (0.9, 0.5)
}

labels = {
    'climate': '気候\n(気温偏差)',
    'trade': '交易\n(難破船数)',
    'population': '人口\n(碑文数)',
    'fiscal': '財政\n(銀含有率)',
    'instability': '不安定\n(指標)'
}

# ノードを描画
for node, (x, y) in nodes.items():
    circle = plt.Circle((x, y), 0.08, color='lightblue', ec='navy', lw=2)
    ax.add_patch(circle)
    ax.text(x, y, labels[node], ha='center', va='center', fontsize=10, fontweight='bold')

# 矢印を描画（グレンジャー検定結果に基づいて色分け）
arrows = [
    ('climate', 'trade'),
    ('trade', 'population'),
    ('population', 'fiscal'),
    ('fiscal', 'instability')
]

for cause, effect in arrows:
    x1, y1 = nodes[cause]
    x2, y2 = nodes[effect]
    
    # グレンジャー検定結果を取得
    try:
        result = analyzer.test_granger_causality(cause, effect)
        is_sig = result['is_significant']
        p_val = result['min_p_value']
        color = 'green' if is_sig else 'gray'
        style = '->' if is_sig else '->'
        lw = 3 if is_sig else 1
    except:
        color = 'gray'
        is_sig = False
        p_val = 1.0
        lw = 1
    
    ax.annotate('', xy=(x2-0.08, y2), xytext=(x1+0.08, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    
    # p値を表示
    mid_x = (x1 + x2) / 2
    ax.text(mid_x, y1 + 0.15, f'p={p_val:.3f}', ha='center', fontsize=9,
           color='green' if is_sig else 'gray')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('因果連鎖の検証結果\n（緑: 有意 p<0.05, 灰: 非有意）', fontsize=14)

plt.tight_layout()
plt.show()

## 10. 図の保存

In [ ]:
# 出力ディレクトリの確認
output_dir = project_root / 'outputs' / 'figures'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"出力ディレクトリ: {output_dir}")

# 時系列プロット
fig = analyzer.plot_time_series()
fig.savefig(output_dir / 'causal_chain_timeseries.png', dpi=300, bbox_inches='tight')
plt.close()
print("保存: causal_chain_timeseries.png")

# 相関ヒートマップ
fig = analyzer.plot_correlation_heatmap()
fig.savefig(output_dir / 'causal_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("保存: causal_correlation_heatmap.png")

# クロス相関
fig = analyzer.plot_cross_correlation()
fig.savefig(output_dir / 'causal_cross_correlation.png', dpi=300, bbox_inches='tight')
plt.close()
print("保存: causal_cross_correlation.png")

print("\n全ての図を保存しました。")

## 11. 考察

### 11.1 分析結果の解釈

因果連鎖 `気候 → 農業 → 人口 → 財政 → 不安定` の各ステップについて:

1. **気候 → 農業**: グレンジャー因果性が認められる場合、気候変動が交易活動（農業生産力の代理指標）に先行して影響を与えていることを示唆

2. **農業 → 人口**: 交易活動の減少が碑文数（人口の代理指標）の減少に先行する場合、経済活動の停滞が人口減少につながることを示唆

3. **人口 → 財政**: 人口減少が銀含有率の低下に先行する場合、税収減による財政悪化を示唆

4. **財政 → 不安定**: 銀含有率の低下が政治不安定に先行する場合、貨幣改悪による社会不安を示唆

### 11.2 限界

- **データの解像度**: 10年単位の集計により、短期的な変動が平滑化されている
- **代理指標の限界**: 難破船数≠農業生産力、碑文数≠人口、など
- **外生ショック**: キプリアヌスの疫病（AD249-262）などの独立した要因が考慮されていない
- **グレンジャー因果性の解釈**: 予測的因果であり、真の因果関係の証明ではない